# Text Feature Extraction — Whisper / BERT / RoBERTa / XML-RoBERTa

| Stage | Model | Output |
|---|---|---|
| Speech-to-text | `WhisperWrapper` | transcript string |
| Token features | `BertWrapper` | `(B, T, 768)` token embeddings |
| Token features | `RobertaWrapper` | `(B, T, 1024)` token embeddings |
| Multilingual sentence features | `XmlRobertaWrapper` | `(B, 768)` sentence embeddings |

`WhisperWrapper` transcribes a 16 kHz mono waveform.  The text is then passed to
the three transformer wrappers.  All wrappers accept a string or list of strings.
Demo uses `multispeaker.wav` as the audio source.

In [1]:
from pathlib import Path

audio_path = Path("../tests/fixtures/audio/multispeaker.wav")
print(f"Audio file: {audio_path}")
print(f"Exists: {audio_path.exists()}")

Audio file: ../tests/fixtures/audio/multispeaker.wav
Exists: True


## 1. Speech-to-Text with Whisper

In [2]:
from exordium.audio.io import load_audio
from exordium.text.whisper import WhisperWrapper

waveform, sr = load_audio(audio_path, target_sample_rate=16000)

whisper = WhisperWrapper(device_id=-1)  # -1 → CPU
text = whisper(waveform)  # waveform: Tensor (T,) → str

print(f"Transcribed text: {text!r}")

/Volumes/UGREEN/Dev/exordium/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

Loading weights:   0%|          | 2/539 [00:00<01:39,  5.41it/s]

Loading weights:  34%|███▎      | 181/539 [00:00<00:00, 479.26it/s]

...

Loading weights: 100%|██████████| 539/539 [00:01<00:00, 453.83it/s]

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Transcribed text: "Hey guys, what do you guys want to eat for lunch? How about Murphy's? They have really good burgers. Vegetarian, do you know if they have any good veggie burgers? You might have a black bean burger, but I'm not really sure what it's like. Okay, looking at Yelp, I can't really tell us any good. Do you want to try Tuk-Tuck? We can get Thai? Hey, I've never been there. I haven't had a lot of Thai food, honestly, so I'm a little unsure what I would order. Yeah, I heard good things about Thai food, too, and I really like Japanese and Korean food, but I really don't know what I would get at a Thai restaurant. Okay, you want to check out taste-based, see if it recommends anything for us? Oh, yeah. Sounds good idea. Oh yeah, they have an awesome looking beef dish, I think I'm really going to enjoy. Yeah, there's a lot of vegetarian options for me. There's also a pork dish, which has a really high match score for me, and it's pretty similar to what I would get at a Korean res

## 1b. Correctness checks — long-form decode, segments, streaming

These assertions live here rather than in CI: they need the **real Whisper weights**, and
correctness is a property of the weights, not of the code. The unit tests build every
wrapper with `pretrained=False` (random weights) and only check input/output contracts,
so they cannot catch a regression in *what the model actually returns*.

Two of the checks below guard bugs that **shipped once already**:

* **long-form decode** — Whisper silently cropped audio to its 30 s window, so everything
  after 30 s of a longer recording was lost (fixed in `2.6.0`).
* **streaming** — streaming a file longer than 30 s first hung forever, then silently
  stopped after the first 30 s chunk.

The fixture (`multispeaker.wav`) is ~61 s, i.e. **longer than Whisper's 30 s window** —
which is the whole point: a 30 s-truncated decode yields ~138 words, the full 61 s yields
250+. Run this notebook after touching `WhisperWrapper` and the cells will fail loudly if
either bug comes back.

In [3]:
duration = waveform.shape[-1] / sr
short = waveform[..., : 10 * sr]  # a 10 s slice, comfortably inside the 30 s window

print(f"fixture duration: {duration:.1f} s  (Whisper's window is 30 s)")
assert duration > 30.0, "fixture must exceed the 30 s window or these checks prove nothing"

fixture duration: 61.2 s  (Whisper's window is 30 s)


In [4]:
# 1) Transcribe from a path, and 2) decode past the 30 s window.
text_from_path = whisper(audio_path)
assert isinstance(text_from_path, str) and text_from_path, "transcription must be a non-empty str"

full_text = whisper(waveform, beam_size=1)
num_words = len(full_text.split())
print(f"words decoded: {num_words}   (30 s-truncated decode would give ~138)")
assert num_words > 180, f"long-form decode regressed: only {num_words} words"

print(f"\nfirst 120 chars: {full_text[:120]!r}")
print(f"last  120 chars: {full_text[-120:]!r}   <- proves the tail past 30 s survived")

words decoded: 257   (30 s-truncated decode would give ~138)

first 120 chars: 'Hey guys, what do you guys want to eat for lunch? How about Murphys? They have really good burgers. Vegetarian, do you k'
last  120 chars: 'ferences, and helps you figure out how to dine with friends. Check us out at mytasteface.com. my taste base.com. Thanks.'   <- proves the tail past 30 s survived


In [5]:
# 3) Segments must span the whole recording, in order, with no blank text.
segments = whisper.transcribe_segments(waveform, beam_size=1)
starts = [s.start for s in segments]

print(f"segments: {len(segments)}   last ends at {segments[-1].end:.1f} s of {duration:.1f} s")
for s in segments[:3]:
    print(f"  [{s.start:6.2f} - {s.end:6.2f}]  {s.text.strip()[:60]!r}")

assert len(segments) > 1, "long audio must produce multiple segments"
assert segments[-1].end > 45.0, "segments stop early — long-form decode regressed"
assert starts == sorted(starts), "segments are not chronological"
assert all(s.text.strip() for s in segments), "a segment came back blank"

# 4) Short audio still segments correctly.
short_segments = whisper.transcribe_segments(short, beam_size=1)
assert len(short_segments) > 0, "short audio produced no segments"
print(f"\n10 s slice -> {len(short_segments)} segment(s)  OK")

segments: 26   last ends at 60.6 s of 61.2 s
  [  0.00 -   2.00]  'Hey guys, what do you guys want to eat for lunch?'
  [  2.00 -   4.00]  'How about Murphys? They have really good burgers.'
  [  4.00 -   7.00]  'Vegetarian, do you know if they have any good veggie burgers'



10 s slice -> 4 segment(s)  OK


In [6]:
# 5) Streaming a short clip, and 6) streaming past the 30 s window.
short_stream = "".join(whisper.stream(short, beam_size=1))
assert len(short_stream.split()) > 5, "streaming a 10 s clip returned almost nothing"
assert "<|" not in short_stream, "timestamp tokens leaked into the streamed text"
print(f"streamed 10 s slice: {len(short_stream.split())} words")

long_stream = "".join(whisper.stream(waveform, beam_size=1))
streamed_words = len(long_stream.split())
print(f"streamed full 61 s : {streamed_words} words   (a 30 s cut-off would give ~138)")
assert streamed_words > 180, f"streaming regressed: only {streamed_words} words"
assert "<|" not in long_stream, "timestamp tokens leaked into the streamed text"

print("\nAll Whisper correctness checks passed.")

streamed 10 s slice: 45 words


streamed full 61 s : 255 words   (a 30 s cut-off would give ~138)

All Whisper correctness checks passed.


## 2. BERT Features

In [7]:
from exordium.text.bert import BertWrapper

bert = BertWrapper(device_id=-1)

# bert() always returns (B, T, 768) token-level tensors
bert_tokens = bert(text)
bert_pooled = bert(text)[:, 0]  # CLS token as sentence-level proxy → (B, 768)

print(f"BERT token-level: shape={tuple(bert_tokens.shape)}  (B x T_tokens x 768)")
print(f"BERT pooled:      shape={tuple(bert_pooled.shape)}  (B x 768)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8209.81it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT token-level: shape=(1, 361, 768)  (B x T_tokens x 768)
BERT pooled:      shape=(1, 768)  (B x 768)


## 3. RoBERTa Features

In [8]:
from exordium.text.roberta import RobertaWrapper

roberta = RobertaWrapper(device_id=-1)

roberta_tokens = roberta(text)
roberta_pooled = roberta(text)[:, 0]  # CLS token as sentence-level proxy → (B, 1024)

print(f"RoBERTa token-level: shape={tuple(roberta_tokens.shape)}  (B x T_tokens x 1024)")
print(f"RoBERTa pooled:      shape={tuple(roberta_pooled.shape)}  (B x 1024)")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7865.29it/s]


[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa token-level: shape=(1, 342, 1024)  (B x T_tokens x 1024)
RoBERTa pooled:      shape=(1, 1024)  (B x 1024)


## 4. XML-RoBERTa Features (Multilingual)

In [9]:
from exordium.text.xml_roberta import XmlRobertaWrapper

xmlr = XmlRobertaWrapper(device_id=-1)

# Accepts a list of texts — returns a batch tensor
sentences = [
    text,  # English (from Whisper)
    "Ceci est un exemple en français.",
    "Dies ist ein Beispiel auf Deutsch.",
]

xmlr_pooled = xmlr(sentences)
print(f"XML-RoBERTa pooled: shape={tuple(xmlr_pooled.shape)}  (3 x 768)")
print("  Each row = one sentence embedding, language-agnostic 768-dim space")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6851.69it/s]

XML-RoBERTa pooled: shape=(3, 768)  (3 x 768)
  Each row = one sentence embedding, language-agnostic 768-dim space


## 5. Single-sentence inference helper

In [10]:
# Use __call__ directly — inference() takes a dict, not a string
bert_vec = bert(text)[0, 0]  # (768,) — CLS token of first sentence
xmlr_vec = xmlr(text)[0]  # (768,) — XmlRoberta pools internally, squeeze batch dim

print(f"bert vector:  shape={tuple(bert_vec.shape)}")
print(f"xmlr vector:  shape={tuple(xmlr_vec.shape)}")

bert vector:  shape=(768,)
xmlr vector:  shape=(768,)
